# ChakraModel — Full Universal Evaluation v9
### All 15 Datasets · Image + Video · Fixed Negative-Control Fallback Bug

**What's new in v9 (fixes from the v8 run):**

- 🐛 **Fixed false-positive bug:** video negative-control (and raw video "detection rate") evaluation was calling `run_yolo_segment(..., fallback=True)`, which forced a full-frame segmentation even when YOLO correctly found nothing. That guaranteed FPR≈1.0 / specificity≈0.0 on `LDPolyp-NoPolyp` and a meaningless 100% "detection rate" on every positive video, regardless of the model's real behavior. Now `fallback=False` for video, so a "detection" only counts when YOLO itself actually fires.
- 🐛 **Fixed CVC-300 skip:** `DATASET_MAP` had the corrected subpaths (`CVC-300/images`, `CVC-300/masks`) but the call site in "Run ALL evaluations" was still hardcoding the old, wrong ones (`CVC-300/Original`, `CVC-300/Ground Truth`). Now pulled from `DATASET_MAP` directly so it can't drift out of sync again.
- 🩹 **Case-insensitive path resolution** (`smart_join`): fixes skips like `PolypDB/BKAI/NBI` failing on an exact-case folder-name mismatch.
- 🔑 **HF_TOKEN support:** reads from Kaggle Secrets automatically (falls back to a manual `os.environ["HF_TOKEN"]` if not set) to avoid the unauthenticated Hugging Face Hub warning.
- 🔍 Added diagnostics for the two still-unexplained skips (`PolypDB/BKAI/NBI`, `PolypGen2021-Video`) so their real folder layout shows up on the next run instead of guessing at a path.

**Carried over from v8:**
- ✅ YOLO fallback (full-image ROI on 0 detections) still used for image datasets with GT masks, where a polyp is always present — that one was fine.
- ✅ Auto-detects all 15 attached datasets by type
- ✅ Full comparison table with per-dataset dice ± std, IoU, detection rate
- ✅ Saves `combined_report_v9.json` + `comparison_table_v9.csv`


In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "ultralytics", "thop", "numpy", "opencv-python",
    "matplotlib", "pandas", "tabulate", "--quiet"], check=False)
print("Packages ready.")
# ── HF_TOKEN setup (v9 fix) ───────────────────────────────────────────────
# Avoids the "unauthenticated requests to HF Hub" warning / rate limits
# when ChakraNet pulls the timm ViT backbone weights.
# Tries Kaggle Secrets first (Add-ons > Secrets in the Kaggle UI), then
# falls back to an env var you set manually in the cell below.
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets ✓")
except Exception as e:
    print(f"Could not load HF_TOKEN from Kaggle Secrets ({e}).")
    # Add your token here:
    os.environ['HF_TOKEN'] = 'REPLACE_WITH_YOUR_HF_TOKEN'
    os.environ['HUGGING_FACE_HUB_TOKEN'] = 'REPLACE_WITH_YOUR_HF_TOKEN'


Packages ready.
HF_TOKEN loaded from Kaggle Secrets ✓


In [2]:
import os, sys, glob, cv2, json, time, shutil, traceback
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

WORKING_DIR = "/kaggle/working"
INPUT_DIR = "/kaggle/input"
print(f"Working dir : {WORKING_DIR}")
print(f"Input dir   : {INPUT_DIR}")
print(f"Datasets attached: {len(os.listdir(INPUT_DIR))}")
for d in sorted(os.listdir(INPUT_DIR)):
    print(f"  /kaggle/input/{d}")


Working dir : /kaggle/working
Input dir   : /kaggle/input
Datasets attached: 2
  /kaggle/input/datasets
  /kaggle/input/models


In [3]:
# =================================================================
# REPLACEMENT FOR CELL 3 — fixed dataset resolution
# =================================================================
# Root cause of the empty results table: /kaggle/input only has ONE
# top-level folder ("datasets"), with everything nested underneath
# as datasets/<owner>/<original-dataset-name>/... . The old
# resolve_dataset() only checked top-level names, so it fuzzy-matched
# almost everything onto "/kaggle/input/datasets" itself (wrong root),
# and true video-only keys never matched at all.
#
# Fix: search the WHOLE tree for a directory literally named ds_key,
# wherever it's nested.

# NOTE: a few of the PolypDB "img_sub"/"mask_sub" values below were
# truncated in the source export this notebook was rebuilt from.
# They are filled in with the best-guess pattern confirmed by the
# In[9] diagnostic cell (PolypDB_center_wise/<CENTER>/<MODALITY>).
# Double check these against your original .ipynb before relying on them.

DATASET_MAP = {
    # ── CODE & WEIGHTS ────────────────────────────────────
    "chakramodel-kaggle-code":     {"role": "code"},
    "finalmuruga-harae":           {"role": "weights"},
    "chakramodel-weightsupdated4": {"role": "weights"},
    "final-om-evlautation-upload": {"role": "eval_output"},
    "om-finalkaggle-upload":       {"role": "eval_output"},

    # ── IMAGE datasets (with GT masks) ──────────────────────
    "endoscene-cvc300-polyp-raw-dataset": {
        "role": "image", "name": "CVC-300",
        # FIXED: real subfolders are "images"/"masks", not
        # "Original"/"Ground Truth" (confirmed from directory tree)
        "img_sub": "CVC-300/images",
        "mask_sub": "CVC-300/masks",
    },
    "chakramodel-evaluation-datasets": {
        "role": "image_multi", "name": "Eval Pack",
        "sub_datasets": [
            {"name": "CVC-ClinicDB", "img_sub": "cvc-clinicdb/images", "mask_sub": "cvc-clinicdb/masks"},
            {"name": "Kvasir-SEG",   "img_sub": "kvasir-seg/images",   "mask_sub": "kvasir-seg/masks"},
            {"name": "ETIS-LARIB",   "img_sub": "etis-larib/images",   "mask_sub": "etis-larib/masks"},
        ],
    },
    "hyperkvasir-dataset-first-half-and-and-ld-dataset": {
        "role": "image_multi", "name": "HyperKvasir+LD Images",
        "sub_datasets": [
            {"name": "HyperKvasir-Seg",
             "img_sub": "hyper-kvasir-segmented-images-part 3/segmented-images/images",
             "mask_sub": "hyper-kvasir-segmented-images-part 3/segmented-images/masks"},
        ],
    },
    "polypdb-polyp-raw": {
        "role": "image_multi", "name": "PolypDB",
        "sub_datasets": [
            {"name": "PolypDB/Simula/NBI",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/NBI/images",   "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/NBI/masks"},
            {"name": "PolypDB/Simula/WLI",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/WLI/images",   "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/WLI/masks"},
            {"name": "PolypDB/BKAI/WLI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI/masks"},
            {"name": "PolypDB/BKAI/NBI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/masks"},
            {"name": "PolypDB/BKAI/BLI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/masks"},
            {"name": "PolypDB/BKAI/FICE",    "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/FICE/images",    "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/FICE/masks"},
            {"name": "PolypDB/BKAI/LCI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/LCI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/LCI/masks"},
            {"name": "PolypDB/Karolinska",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Karolinska/images",   "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Karolinska/masks"},
            {"name": "PolypDB/FICE",         "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/FICE/images",       "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/FICE/masks"},
            {"name": "PolypDB/BLI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/BLI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/BLI/masks"},
            {"name": "PolypDB/NBI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/NBI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/NBI/masks"},
            {"name": "PolypDB/WLI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/WLI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/WLI/masks"},
            {"name": "PolypDB/LCI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/LCI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/LCI/masks"},
        ],
    },

    # ── VIDEO datasets ────────────────────────────────────
    # NOTE: these 3 keys were NOT found anywhere in the directory
    # tree under these exact names. Either they weren't attached to
    # this kernel, or they're attached under a different slug. If
    # you re-attach them, keep the key here matching the exact
    # folder name Kaggle mounts it as.
    "polypgen20021-video": {
        "role": "video_positive", "name": "PolypGen2021-Video",
        "note": "Polyps present — dice/IoU per frame",
    },
    "ldpolypvideowithoutpolyps": {
        "role": "video_negative", "name": "LDPolyp-NoPolyp",
        "note": "NO polyps — specificity check (false positive rate)",
    },
    "ldpolypvideopolyponly": {
        "role": "video_positive", "name": "LDPolyp-PolypOnly",
        "note": "Polyps only — sensitivity check (dice/IoU per frame)",
    },
    # These 3 matched exactly in the tree — no changes needed.
    "hperkvasir-labeled-videos-part2-002": {
        "role": "video_positive", "name": "HyperKvasir-Video-P2B",
        "note": "HyperKvasir labeled video frames part 2 (B)",
    },
    "hyperkvasir-labeled-videos-part2-001": {
        "role": "video_positive", "name": "HyperKvasir-Video-P2A",
        "note": "HyperKvasir labeled video frames part 2 (A)",
    },
    "cvc-sample-video": {
        "role": "video_positive", "name": "CVC-Video",
        "note": "CVC sample video clips",
    },
}


def resolve_dataset(ds_key, ds_cfg):
    """Exact-name recursive search, anywhere under INPUT_DIR."""
    for root, dirs, files in os.walk(INPUT_DIR):
        if ds_key in dirs:
            return os.path.join(root, ds_key)
    return None


def smart_join(root, relpath):
    """
    Join root + relpath, but resolve each path component case-insensitively
    if the exact-case component doesn't exist. Fixes skips like
    'PolypDB_center_wise/BKAI/NBI/images' when the real folder on disk is
    'Nbi' / 'nbi' / etc. Falls back to the plain (uncorrected) path if no
    case-insensitive match is found either, so callers still get a useful
    "not found" message pointing at the path that was actually tried.
    """
    cur = root
    for part in Path(relpath).parts:
        candidate = os.path.join(cur, part)
        if os.path.isdir(candidate) or os.path.isfile(candidate):
            cur = candidate
            continue
        if os.path.isdir(cur):
            match = next((e for e in os.listdir(cur) if e.lower() == part.lower()), None)
            if match:
                cur = os.path.join(cur, match)
                continue
        cur = candidate  # give up — keep the literal (uncorrected) path
    return cur


RESOLVED = {}
print("\nDataset resolution:")
found_count = 0
for key, cfg in DATASET_MAP.items():
    path = resolve_dataset(key, cfg)
    RESOLVED[key] = path
    role = cfg["role"]
    status = "✓ found" if path else "✗ not found"
    if path:
        found_count += 1
    print(f"  [{role:15s}] {key:50s} {status}")

print(f"\n{found_count}/{len(DATASET_MAP)} dataset keys resolved to a real folder.")

# =================================================================
# ADD THIS BEFORE CELL 4 — diagnose which chakranet_segmenter.py
# actually defines ChakraNetSegmenter, across all attached copies.
# =================================================================
import re

print("\nSearching all attached copies of chakranet_segmenter.py for class names...")
for root, dirs, files in os.walk(INPUT_DIR):
    if "chakranet_segmenter.py" in files:
        fpath = os.path.join(root, "chakranet_segmenter.py")
        with open(fpath, "r", errors="ignore") as f:
            src = f.read()
        classes = re.findall(r"^class\s+(\w+)", src, re.MULTILINE)
        print(f"  {fpath}")
        print(f"    classes defined: {classes}")

# Once you see which copy actually defines ChakraNetSegmenter (or
# whatever it's now called), point CODE_PATH at that exact folder
# in Cell 4 instead of relying on os.walk's first hit, e.g.:
#
# CODE_PATH = resolve_dataset("final-om-evlautation-upload", {})
# (or whichever key's copy has the right class)



Dataset resolution:
  [code           ] chakramodel-kaggle-code                           ✓ found
  [weights        ] finalmuruga-harae                                 ✓ found
  [weights        ] chakramodel-weightsupdated4                       ✓ found
  [eval_output    ] final-om-evlautation-upload                       ✓ found
  [eval_output    ] om-finalkaggle-upload                             ✓ found
  [image          ] endoscene-cvc300-polyp-raw-dataset                ✓ found
  [image_multi    ] chakramodel-evaluation-datasets                   ✓ found
  [image_multi    ] hyperkvasir-dataset-first-half-and-and-ld-dataset ✓ found
  [image_multi    ] polypdb-polyp-raw                                 ✓ found
  [video_positive ] polypgen20021-video                               ✓ found
  [video_negative ] ldpolypvideowithoutpolyps                         ✓ found
  [video_positive ] ldpolypvideopolyponly                             ✓ found
  [video_positive ] hperkvasir-labeled-vide

In [4]:
# ────────────────────────────────────────────
# Load YOLO + ChakraNet Transformer
# ────────────────────────────────────────────

# 1. Find & copy source code
CODE_PATH = None
for root, dirs, files in os.walk(INPUT_DIR):
    if "chakranet_segmenter.py" in files and os.path.basename(root) == "src":
        CODE_PATH = os.path.dirname(root)
        break

if CODE_PATH:
    shutil.copytree(CODE_PATH, f"{WORKING_DIR}/chakramodel", dirs_exist_ok=True)
    os.chdir(f"{WORKING_DIR}/chakramodel")
    if f"{WORKING_DIR}/chakramodel/src" not in sys.path:
        sys.path.insert(0, f"{WORKING_DIR}/chakramodel/src")
    print(f"Source code loaded from: {CODE_PATH}")
else:
    print("WARNING: chakranet_segmenter.py not found — segmenter will be mocked.")

# 2. Find weights
os.makedirs(f"{WORKING_DIR}/chakramodel/weights", exist_ok=True)

TRANSFORMER_PTH = None
YOLO_PT         = None

for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f == "chakra_transformer_best.pth" and TRANSFORMER_PTH is None:
            TRANSFORMER_PTH = os.path.join(root, f)
        if f == "best.pt" and YOLO_PT is None:
            YOLO_PT = os.path.join(root, f)

if TRANSFORMER_PTH:
    shutil.copy(TRANSFORMER_PTH, f"{WORKING_DIR}/chakramodel/weights/chakra_transformer_best.pth")
    print(f"Transformer weights: {TRANSFORMER_PTH}")
else:
    print("WARNING: chakra_transformer_best.pth not found.")

if YOLO_PT:
    shutil.copy(YOLO_PT, f"{WORKING_DIR}/chakramodel/weights/best.pt")
    YOLO_PT_LOCAL = f"{WORKING_DIR}/chakramodel/weights/best.pt"
    print(f"YOLO weights: {YOLO_PT}")
else:
    print("WARNING: best.pt not found.")
    YOLO_PT_LOCAL = None

# 3. Load YOLO on GPU 0
YOLO_MODEL = None
if YOLO_PT_LOCAL:
    try:
        from ultralytics import YOLO
        # Explicitly move YOLO to GPU 0
        YOLO_MODEL = YOLO(YOLO_PT_LOCAL).to("cuda:0")
        print("YOLO loaded on cuda:0 ✓")
    except Exception as e:
        print(f"YOLO load failed: {e}")

# 4. Load ChakraNet segmenter on GPU 1
SEG_MODEL = None
try:
    from chakranet_segmenter import ChakraNet
    import torch

    torch.backends.cudnn.benchmark = True  # Optimize CUDA kernels

    # Check if a second GPU exists, otherwise fallback to GPU 0
    DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

    SEG_MODEL = ChakraNet(
        device=DEVICE,
        img_size=(384, 384),
        weights_path=f"{WORKING_DIR}/chakramodel/weights/chakra_transformer_best.pth"
    )
    print(f"ChakraNet loaded on {DEVICE} ✓")

except Exception as e:
    print(f"Segmenter load failed: {e}")


Source code loaded from: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code
Transformer weights: /kaggle/input/datasets/gokulrocky/chakratransformer-weights/chakra_transformer_best.pth
YOLO weights: /kaggle/input/datasets/gokulrocky/finalmuruga-harae/weights/best.pt
Creating new Ultralytics Settings v0.0.8 file ✅
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
YOLO loaded on cuda:0 ✓
[INFO] ChakraNet: Loaded weights from /kaggle/working/chakramodel/weights/chakra_transformer_best.pth
ChakraNet loaded on cuda:1 ✓


In [5]:
_SEGMENT_ERROR_LOGGED = False  # set True after the first logged segment_roi() error

# ────────────────────────────────────────────
# Core evaluation utilities
# ────────────────────────────────────────────

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
VID_EXTS = {".mp4", ".avi", ".mov", ".mkv"}


def dice_iou(pred_bin, gt_bin):
    pred = pred_bin.astype(bool)
    gt   = gt_bin.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    dice = 2 * inter / (pred.sum() + gt.sum() + 1e-8)
    iou  = inter / (union + 1e-8)
    return float(dice), float(iou)


def run_yolo_segment(img_bgr, yolo_model, seg_model, fallback=True):
    """
    Run YOLO detect → crop → segment_roi().
    If YOLO finds nothing and fallback=True, uses full image as ROI.
    Returns (pred_mask_hw, n_detections, mode).
    """
    h, w = img_bgr.shape[:2]
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    mode      = "none"

    # YOLO
    boxes = []
    if yolo_model is not None:
        try:
            # Force YOLO inference to run on GPU 0
            res = yolo_model(img_bgr, device=0, verbose=False)
            if len(res) > 0 and res[0].boxes is not None and len(res[0].boxes) > 0:
                boxes = res[0].boxes.xyxy.cpu().numpy().tolist()
        except Exception:
            pass

    if len(boxes) == 0:
        if fallback:
            boxes = [[0, 0, w, h]]
            mode = "fallback"
        else:
            return pred_mask, 0, "no_detection"
    else:
        mode = "yolo"

    if seg_model is None:
        # Mock: return empty mask (for testing pipeline without weights)
        return pred_mask, len(boxes), mode + "_mock"

    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = img_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        try:
            crop_mask = seg_model.segment_roi(crop)[0]
            if crop_mask is None:
                continue

            crop_h, crop_w = y2 - y1, x2 - x1
            if crop_mask.shape != (crop_h, crop_w):
                crop_mask = cv2.resize(crop_mask.astype(np.uint8), (crop_w, crop_h))

            pred_mask[y1:y2, x1:x2] = np.maximum(pred_mask[y1:y2, x1:x2], crop_mask)

        except Exception as ex:
            # FIXED: this used to silently swallow every exception...
            # exactly how the whole pipeline ran on a mock/empty mask without
            # any visible warning before. Log the first occurrence so a broken
            # segment_roi() call (wrong API, shape mismatch, etc.) is visible.
            global _SEGMENT_ERROR_LOGGED
            if not _SEGMENT_ERROR_LOGGED:
                print(f"  [run_yolo_segment] segment_roi() raised: {ex}")
                _SEGMENT_ERROR_LOGGED = True

    return pred_mask, len(boxes), mode


def find_mask_for_image(img_path, mask_root):
    """Find the GT mask file for a given image path."""
    stem = Path(img_path).stem
    for ext in [".png", ".jpg", ".bmp", ".tif"]:
        candidate = os.path.join(mask_root, stem + ext)
        if os.path.exists(candidate):
            return candidate
    # Search recursively
    for root, _, files in os.walk(mask_root):
        for f in files:
            if Path(f).stem == stem:
                return os.path.join(root, f)
    return None


def auto_find_img_mask_dirs(dataset_root):
    """
    Auto-detect images/ and masks/ subdirectories in a dataset root.
    Returns list of (img_dir, mask_dir, label) tuples.
    """
    found = []
    for root, dirs, files in os.walk(dataset_root):
        img_exts_here = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
        if img_exts_here < 3:
            continue
        # Look for sibling mask directory
        parent = os.path.dirname(root)
        base   = os.path.basename(root).lower()
        if any(k in base for k in ("image", "img", "frame", "original", "photo")):
            mask_candidates = ["masks", "mask", "gt", "ground_truth",
                               "groundtruth", "annotation", "ann", "label"]
            for mc in mask_candidates:
                mp = os.path.join(parent, mc)
                if os.path.isdir(mp):
                    found.append((root, mp, os.path.basename(parent)))
                    break
    return found if found else []


print("Evaluation utilities loaded ✓")


Evaluation utilities loaded ✓


In [6]:
import numpy as np
# Create a dummy image
dummy_input = np.zeros((448, 448, 3), dtype=np.uint8)
out = SEG_MODEL.segment_roi(dummy_input)
print("Tuple length:", len(out))

# Assuming the main mask is the first element
primary_mask = out[0]
print("Primary Mask Type:", type(primary_mask))
print("Primary Mask Shape:", primary_mask.shape if hasattr(primary_mask, 'shape') else "N/A")
print("Min:", primary_mask.min() if hasattr(primary_mask, 'min') else "N/A")
print("Max:", primary_mask.max() if hasattr(primary_mask, 'max') else "N/A")


Tuple length: 4
Primary Mask Type: <class 'numpy.ndarray'>
Primary Mask Shape: (448, 448)
Min: 0
Max: 0


In [7]:
# ────────────────────────────────────────────
# Image dataset evaluator
# ────────────────────────────────────────────

def evaluate_image_dataset(name, img_dir, mask_dir, max_images=500):
    """
    Evaluate ChakraModel on an image segmentation dataset.
    Returns a result dict.
    """
    if not os.path.isdir(img_dir):
        return {"name": name, "status": "skip", "reason": f"img_dir not found: {img_dir}"}
    if not os.path.isdir(mask_dir):
        return {"name": name, "status": "skip", "reason": f"mask_dir not found: {mask_dir}"}

    img_files = sorted([
        f for f in os.listdir(img_dir)
        if Path(f).suffix.lower() in IMG_EXTS
    ])[:max_images]

    if not img_files:
        return {"name": name, "status": "skip", "reason": "no image files found"}

    dices, ious = [], []
    n_yolo, n_fallback, n_skip = 0, 0, 0
    per_image = []

    for fname in img_files:
        img_path  = os.path.join(img_dir, fname)
        mask_path = find_mask_for_image(img_path, mask_dir)
        if mask_path is None:
            n_skip += 1
            continue

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            n_skip += 1
            continue
        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            n_skip += 1
            continue

        pred_mask, n_boxes, mode = run_yolo_segment(
            img_bgr, YOLO_MODEL, SEG_MODEL, fallback=True
        )

        gt_bin   = (gt > 127).astype(np.uint8)
        pred_bin = (pred_mask > 127).astype(np.uint8)
        d, iou = dice_iou(pred_bin, gt_bin)
        dices.append(d)
        ious.append(iou)
        if "fallback" in mode: n_fallback += 1
        elif "yolo" in mode:   n_yolo += 1
        per_image.append({"file": fname, "dice": round(d,4), "iou": round(iou,4), "mode": mode})

    if not dices:
        return {"name": name, "status": "no_valid_pairs"}

    return {
        "name":           name,
        "type":           "image",
        "status":         "done",
        "n_total":        len(img_files),
        "n_evaluated":    len(dices),
        "n_skip":         n_skip,
        "dice_mean":      round(float(np.mean(dices)), 4),
        "dice_std":       round(float(np.std(dices)),  4),
        "dice_median":    round(float(np.median(dices)),4),
        "iou_mean":       round(float(np.mean(ious)),  4),
        "iou_std":        round(float(np.std(ious)),   4),
        "yolo_detections": n_yolo,
        "fallback_used":  n_fallback,
        "per_image":      per_image,
    }


print("Image evaluator ready ✓")


Image evaluator ready ✓


In [8]:
# ────────────────────────────────────────────
# Video dataset evaluator
# ────────────────────────────────────────────

def evaluate_video_dataset(name, dataset_root, is_negative=False,
                            max_videos=20, max_frames_per_video=150, sample_every=5):
    """
    Evaluate ChakraModel on a video dataset.
    - is_negative=True  → no GT masks, compute false-positive rate
    - is_negative=False → look for GT masks per frame, compute dice/IoU
    """
    if not os.path.isdir(dataset_root):
        return {"name": name, "status": "skip", "reason": f"root not found: {dataset_root}"}

    # Find video files
    video_files = []
    for ext in VID_EXTS:
        video_files += glob.glob(os.path.join(dataset_root, f"**/*{ext}"), recursive=True)
    video_files = sorted(video_files)[:max_videos]

    # Also find frame image directories (some video datasets are pre-extracted)
    frame_dirs = auto_find_img_mask_dirs(dataset_root)

    dices, ious = [], []
    fp_frames, total_frames = 0, 0
    n_videos_processed = 0

    # ── Pre-extracted frames mode ─────────────────────────
    if frame_dirs and not is_negative:
        for img_dir, mask_dir, label in frame_dirs:
            res = evaluate_image_dataset(
                f"{name}/{label}", img_dir, mask_dir,
                max_images=max_frames_per_video
            )
            if res.get("status") == "done":
                dices += [p["dice"] for p in res["per_image"]]
                ious  += [p["iou"]  for p in res["per_image"]]
                n_videos_processed += 1

        if dices:
            return {
                "name": name, "type": "video_frames",
                "status": "done", "mode": "pre_extracted_frames",
                "n_frame_sets":  n_videos_processed,
                "n_frames_total": len(dices),
                "dice_mean":  round(float(np.mean(dices)), 4),
                "dice_std":   round(float(np.std(dices)),  4),
                "dice_median":round(float(np.median(dices)),4),
                "iou_mean":   round(float(np.mean(ious)),  4),
                "iou_std":    round(float(np.std(ious)),   4),
                "is_negative": False,
            }

    # ── Raw video mode ────────────────────────────────
    if not video_files:
        # Last resort: treat root as flat image directory
        img_files = [f for f in os.listdir(dataset_root)
                     if Path(f).suffix.lower() in IMG_EXTS]
        if img_files:
            return evaluate_image_dataset(
                name, dataset_root, dataset_root,  # no separate mask dir
                max_images=max_frames_per_video
            )
        return {"name": name, "status": "skip", "reason": "no video or image files found"}

    for vid_path in video_files:
        cap = cv2.VideoCapture(vid_path)
        if not cap.isOpened():
            continue
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fid = 0
        while fid < min(total, max_frames_per_video * sample_every):
            ret, frame = cap.read()
            if not ret:
                break
            if fid % sample_every == 0:
                # v9 fix: fallback=False here. With fallback=True, YOLO finding
                # nothing (the CORRECT outcome on a no-polyp frame) was silently
                # replaced by "segment the whole frame anyway", which almost
                # always produces some nonzero mask pixels -> guaranteed false
                # positive on negative controls, and a meaningless 100%
                # "detection rate" on positive videos. Only count it as a
                # detection if YOLO itself actually fired.
                pred_mask, n_boxes, mode = run_yolo_segment(
                    frame, YOLO_MODEL, SEG_MODEL, fallback=False
                )
                pred_bin = (pred_mask > 127)
                total_frames += 1
                if is_negative:
                    if pred_bin.any():
                        fp_frames += 1
                else:
                    # No GT mask available for raw video → record detection rate
                    if pred_bin.any():
                        dices.append(1.0)  # placeholder; no GT available
                    else:
                        dices.append(0.0)
            fid += 1
        cap.release()
        n_videos_processed += 1

    if is_negative:
        fpr = round(fp_frames / max(total_frames, 1), 4)
        return {
            "name": name, "type": "video_negative",
            "status": "done",
            "n_videos": n_videos_processed,
            "n_frames_sampled": total_frames,
            "false_positive_frames": fp_frames,
            "false_positive_rate": fpr,
            "specificity": round(1.0 - fpr, 4),
            "is_negative": True,
        }

    return {
        "name": name, "type": "video_positive",
        "status": "done",
        "n_videos": n_videos_processed,
        "n_frames_sampled": total_frames,
        "detection_rate": round(np.mean(dices), 4) if dices else 0.0,
        "note": "No GT masks found for raw video — detection rate reported instead of dice/IoU",
        "is_negative": False,
    }


print("Video evaluator ready ✓")


Video evaluator ready ✓


In [9]:
import os

# Check CVC-300 structure
cvc_path = "/kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset"
print(f"Contents of CVC-300 root: {os.listdir(cvc_path)}")
cvc_sub = os.path.join(cvc_path, "CVC-300")
if os.path.isdir(cvc_sub):
    print(f"Contents of CVC-300/CVC-300: {os.listdir(cvc_sub)}")

# Check PolypDB structure
polypdb_path = "/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw"
print(f"Contents of PolypDB root: {os.listdir(polypdb_path)}")

# ── v9 diagnostics: figure out the two unresolved SKIPs from the last run ──
# 1. PolypDB/BKAI/NBI — img_dir not found at .../PolypDB_center_wise/BKAI/NBI/images
bkai_root = os.path.join(
    polypdb_path,
    "PolypDB/PolypDB/PolypDB_center_wise/BKAI"
)
if os.path.isdir(bkai_root):
    print(f"\nContents of PolypDB_center_wise/BKAI: {os.listdir(bkai_root)}")
    nbi_candidates = [d for d in os.listdir(bkai_root) if "nbi" in d.lower()]
    print(f"  NBI-like subfolders found: {nbi_candidates}")
    for c in nbi_candidates:
        print(f"    {c} -> {os.listdir(os.path.join(bkai_root, c))}")
else:
    print(f"\nBKAI root not found at all: {bkai_root}")

# 2. PolypGen2021-Video — resolved a root folder but "no video or image files found"
polypgen_root = RESOLVED.get("polypgen20021-video")
print(f"\nPolypGen2021-Video resolved root: {polypgen_root}")
if polypgen_root and os.path.isdir(polypgen_root):
    for root, dirs, files in os.walk(polypgen_root):
        depth = root[len(polypgen_root):].count(os.sep)
        if depth <= 2:
            print(f"  {root} -> {len(files)} files, {dirs[:5]}{'...' if len(dirs) > 5 else ''}")


Contents of CVC-300 root: ['CVC-300']
Contents of CVC-300/CVC-300: ['images', 'masks']
Contents of PolypDB root: ['PolypDB']

Contents of PolypDB_center_wise/BKAI: ['FICE', 'BLI', 'WLI', 'LCI']
  NBI-like subfolders found: []

PolypGen2021-Video resolved root: /kaggle/input/datasets/gokulraj324/polypgen20021-video
  /kaggle/input/datasets/gokulraj324/polypgen20021-video -> 0 files, ['PolypGen2021_MultiCenterData_v3']
  /kaggle/input/datasets/gokulraj324/polypgen20021-video/PolypGen2021_MultiCenterData_v3 -> 6 files, ['data_C5', 'sequenceData', 'codes', 'dataDetails_PolypGen_SingleFrames', 'data_C6']...
  /kaggle/input/datasets/gokulraj324/polypgen20021-video/PolypGen2021_MultiCenterData_v3/data_C5 -> 0 files, ['bbox_C5', 'masks_C5', 'bbox_image_C5', 'images_C5']
  /kaggle/input/datasets/gokulraj324/polypgen20021-video/PolypGen2021_MultiCenterData_v3/sequenceData -> 0 files, ['negativeOnly', 'positive']
  /kaggle/input/datasets/gokulraj324/polypgen20021-video/PolypGen2021_MultiCenterDat

In [10]:
# ────────────────────────────────────────────
# Run ALL evaluations
# ────────────────────────────────────────────

ALL_RESULTS = []


def run_image_ds(name, ds_key, img_sub, mask_sub):
    root = RESOLVED.get(ds_key)
    if not root:
        print(f"  SKIP {name}: dataset not attached")
        return
    # v9 fix: smart_join tolerates case mismatches (e.g. "NBI" vs "Nbi")
    # instead of failing the whole lookup on an exact-case miss.
    img_dir  = smart_join(root, img_sub)
    mask_dir = smart_join(root, mask_sub)
    print(f"  Evaluating {name} ...")
    r = evaluate_image_dataset(name, img_dir, mask_dir)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        print(f"    dice={r['dice_mean']:.4f}±{r['dice_std']:.4f}  iou={r['iou_mean']:.4f}  n={r['n_evaluated']}")
    else:
        print(f"    SKIP: {r.get('reason','')}")


def run_video_ds(name, ds_key, is_neg=False):
    root = RESOLVED.get(ds_key)
    if not root:
        print(f"  SKIP {name}: dataset not attached")
        return
    print(f"  Evaluating {name} ({'negative control' if is_neg else 'positive'}) ...")
    r = evaluate_video_dataset(name, root, is_negative=is_neg)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        if is_neg:
            print(f"    FPR={r['false_positive_rate']:.4f}  specificity={r['specificity']:.4f}")
        else:
            d = r.get('dice_mean')
            if d is not None:
                print(f"    dice={d:.4f}±{r.get('dice_std',0):.4f}  iou={r.get('iou_mean',0):.4f}")
            else:
                print(f"    detection_rate={r.get('detection_rate',0):.4f}  (no GT masks)")
    else:
        print(f"    SKIP: {r.get('reason','')}")


print("=" * 60)
print("IMAGE DATASETS")
print("=" * 60)

# CVC-300
_cvc_cfg = DATASET_MAP["endoscene-cvc300-polyp-raw-dataset"]
run_image_ds("CVC-300",
             "endoscene-cvc300-polyp-raw-dataset",
             _cvc_cfg["img_sub"], _cvc_cfg["mask_sub"])

# CVC-ClinicDB, Kvasir-SEG, ETIS-LARIB
for sub in DATASET_MAP["chakramodel-evaluation-datasets"]["sub_datasets"]:
    run_image_ds(sub["name"], "chakramodel-evaluation-datasets",
                 sub["img_sub"], sub["mask_sub"])

# HyperKvasir
run_image_ds("HyperKvasir-Seg",
             "hyperkvasir-dataset-first-half-and-and-ld-dataset",
             "hyper-kvasir-segmented-images-part 3/segmented-images/images",
             "hyper-kvasir-segmented-images-part 3/segmented-images/masks")

# PolypDB
for sub in DATASET_MAP["polypdb-polyp-raw"]["sub_datasets"]:
    run_image_ds(sub["name"], "polypdb-polyp-raw", sub["img_sub"], sub["mask_sub"])

print("\n" + "=" * 60)
print("VIDEO DATASETS")
print("=" * 60)

run_video_ds("PolypGen2021-Video",   "polypgen20021-video",                is_neg=False)
run_video_ds("LDPolyp-NoPolyp",      "ldpolypvideowithoutpolyps",          is_neg=True)
run_video_ds("LDPolyp-PolypOnly",    "ldpolypvideopolyponly",              is_neg=False)
run_video_ds("HyperKvasir-Video-P2A","hyperkvasir-labeled-videos-part2-001", is_neg=False)
run_video_ds("HyperKvasir-Video-P2B","hperkvasir-labeled-videos-part2-002",  is_neg=False)
run_video_ds("CVC-Video",            "cvc-sample-video",                   is_neg=False)

print("\n✓ All evaluations complete.")


IMAGE DATASETS
  Evaluating CVC-300 ...
    dice=0.4334±0.3380  iou=0.3439  n=60
  Evaluating CVC-ClinicDB ...
    dice=0.6855±0.3036  iou=0.5918  n=495
  Evaluating Kvasir-SEG ...
    dice=0.8505±0.1778  iou=0.7715  n=500
  Evaluating ETIS-LARIB ...
    dice=0.9714±0.0008  iou=0.9443  n=5
  Evaluating HyperKvasir-Seg ...
    dice=0.8540±0.1787  iou=0.7768  n=500
  Evaluating PolypDB/Simula/NBI ...
    dice=0.6410±0.3052  iou=0.5389  n=146
  Evaluating PolypDB/Simula/WLI ...
    dice=0.7301±0.2746  iou=0.6359  n=500
  Evaluating PolypDB/BKAI/WLI ...
    dice=0.8419±0.2198  iou=0.7710  n=500
  Evaluating PolypDB/BKAI/NBI ...
    SKIP: img_dir not found: /kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/images
  Evaluating PolypDB/BKAI/BLI ...
    dice=0.6228±0.3302  iou=0.5299  n=70
  Evaluating PolypDB/BKAI/FICE ...
    dice=0.7194±0.3142  iou=0.6403  n=70
  Evaluating PolypDB/BKAI/LCI ...
    dice=0.7530±0.2706  iou=0.6682  n=60
  Evaluat

In [11]:
# ────────────────────────────────────────────
# Comprehensive comparison table
# ────────────────────────────────────────────

import pandas as pd

rows = []
for r in ALL_RESULTS:
    if r.get("status") != "done":
        continue

    is_neg  = r.get("is_negative", False)
    ds_type = r.get("type", "unknown")

    if is_neg:
        rows.append({
            "Dataset":      r["name"],
            "Type":         "video-negative",
            "N Samples":    r.get("n_frames_sampled", "-"),
            "Dice Mean":    "N/A (negative ctrl)",
            "Dice Std":     "-",
            "IoU Mean":     "-",
            "Specificity":  f"{r['specificity']:.4f}",
            "FP Rate":      f"{r['false_positive_rate']:.4f}",
            "YOLO Detect%": "-",
            "Verdict":      "✓ PASS" if r['specificity'] >= 0.90 else "✗ FAIL",
        })
    elif "dice_mean" in r:
        yolo_pct = "-"
        if "yolo_detections" in r and "n_evaluated" in r and r["n_evaluated"] > 0:
            yolo_pct = f"{100*r['yolo_detections']/r['n_evaluated']:.0f}%"
        verdict = "✓ PASS" if r["dice_mean"] >= 0.60 else ("~ WEAK" if r["dice_mean"] >= 0.30 else "✗ FAIL")
        rows.append({
            "Dataset":      r["name"],
            "Type":         ds_type,
            "N Samples":    r.get("n_evaluated", r.get("n_frames_total", "-")),
            "Dice Mean":    f"{r['dice_mean']:.4f}",
            "Dice Std":     f"{r['dice_std']:.4f}",
            "IoU Mean":     f"{r['iou_mean']:.4f}",
            "Specificity":  "-",
            "FP Rate":      "-",
            "YOLO Detect%": yolo_pct,
            "Verdict":      verdict,
        })
    else:
        rows.append({
            "Dataset":      r["name"],
            "Type":         ds_type,
            "N Samples":    r.get("n_frames_sampled", "-"),
            "Dice Mean":    "-",
            "Dice Std":     "-",
            "IoU Mean":     "-",
            "Specificity":  "-",
            "FP Rate":      "-",
            "YOLO Detect%": f"{100*r.get('detection_rate',0):.0f}%",
            "Verdict":      "~ INFO (no GT masks)",
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 90)
print("CHAKRAMODEL v9 — FULL EVALUATION COMPARISON TABLE")
print("=" * 90)
print(df.to_string(index=False))
print("=" * 90)

# Summary stats
image_rows = [r for r in ALL_RESULTS if r.get("status")=="done" and "dice_mean" in r and not r.get("is_negative")]
if image_rows:
    all_dice = [r["dice_mean"] for r in image_rows]
    print(f"\nImage datasets summary:")
    print(f"  Overall mean Dice : {np.mean(all_dice):.4f}")
    print(f"  Best dataset      : {max(image_rows, key=lambda x: x['dice_mean'])['name']} ({max(all_dice):.4f})")
    print(f"  Worst dataset     : {min(image_rows, key=lambda x: x['dice_mean'])['name']} ({min(all_dice):.4f})")

df.to_csv(f"{WORKING_DIR}/comparison_table_v9.csv", index=False)
print(f"\nSaved: {WORKING_DIR}/comparison_table_v9.csv")



CHAKRAMODEL v9 — FULL EVALUATION COMPARISON TABLE
               Dataset            Type N Samples            Dice Mean Dice Std IoU Mean Specificity FP Rate YOLO Detect%              Verdict
               CVC-300           image         60               0.4334   0.3380   0.3439           -       -          93%               ~ WEAK
           CVC-ClinicDB           image        495               0.6855   0.3036   0.5918           -       -         100%               ✓ PASS
             Kvasir-SEG           image        500               0.8505   0.1778   0.7715           -       -          99%               ✓ PASS
             ETIS-LARIB           image          5               0.9714   0.0008   0.9443           -       -         100%               ✓ PASS
        HyperKvasir-Seg           image        500               0.8540   0.1787   0.7768           -       -          99%               ✓ PASS
     PolypDB/Simula/NBI           image        146               0.6410   0.3052   0.538

In [12]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

image_rows = [r for r in ALL_RESULTS
              if r.get("status") == "done" and "dice_mean" in r]

if image_rows:
    names  = [r["name"].split("/")[-1] for r in image_rows]
    means  = [r["dice_mean"] for r in image_rows]
    stds   = [r["dice_std"]  for r in image_rows]
    colors = ["#2ecc71" if m >= 0.60 else "#e67e22" if m >= 0.30 else "#e74c3c" for m in means]

    fig, ax = plt.subplots(figsize=(max(12, len(names)*0.7), 6))
    bars = ax.bar(names, means, yerr=stds, capsize=4,
                  color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.axhline(0.60, color="#2ecc71", linestyle="--", linewidth=1.2, label="Good threshold")
    ax.axhline(0.30, color="#e74c3c", linestyle="--", linewidth=1.2, label="Poor threshold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Dice Score")
    ax.set_title("ChakraModel v9 — Dice Score per Dataset", fontsize=13, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    p1 = mpatches.Patch(color="#2ecc71", alpha=0.85, label="≥70.60 PASS")
    p2 = mpatches.Patch(color="#e67e22", alpha=0.85, label="0.30-0.60 WEAK")
    p3 = mpatches.Patch(color="#e74c3c", alpha=0.85, label="<0.30 FAIL")
    ax.legend(handles=[p1, p2, p3], loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{WORKING_DIR}/dice_comparison_v9.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved.")


Plot saved.


In [13]:
import json as _json

report = {
    "version": "v9",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "n_datasets_evaluated": len([r for r in ALL_RESULTS if r.get("status")=="done"]),
    "results": ALL_RESULTS,
}

out = f"{WORKING_DIR}/combined_report_v9.json"
with open(out, "w") as f:
    _json.dump(report, f, indent=2, default=str)
print(f"Full report saved: {out}")

# Zip artifacts
os.chdir(WORKING_DIR)
os.system("zip -r chakra_v9_artifacts.zip combined_report_v9.json comparison_table_v9.csv dice_comparison_v9.png")
print("Done! Download chakra_v9_artifacts.zip from the output panel.")


Full report saved: /kaggle/working/combined_report_v9.json
Done! Download chakra_v9_artifacts.zip from the output panel.


---
## v10 PATCH — PolypGen2021-Video real frame-mask pairs

Confirmed against an actual `tree` export of `/kaggle/input` (not just directory listings):

- `PolypGen2021_MultiCenterData_v3/sequenceData/positive/seq{1..23}/images_seq{N}` + `masks_seq{N}` — 23 sequences, each a real image+mask pair (unused by v9 because `auto_find_img_mask_dirs()` only matches mask folders named exactly `masks`/`mask`/`gt`/etc., not suffixed names like `masks_seq1`).
- `PolypGen2021_MultiCenterData_v3/data_C{1..6}/images_C{N}` + `masks_C{N}` — same suffix problem, 6 more image+mask pairs.
- `PolypGen2021_MultiCenterData_v3/sequenceData/negativeOnly/seq{1..23}_neg/` — flat `.jpg` frames, **no masks**, no polyps — a genuine negative control, just never wired up.

This patch adds explicit `sub_datasets` entries for all of these (reusing your existing `run_image_ds` / `evaluate_image_dataset`) and a small negative-control evaluator for the mask-less negative frames. Run this after Cell 10 (or instead of skipping `PolypGen2021-Video`) — it appends its results into the same `ALL_RESULTS` list so Cell 11's comparison table picks them up automatically if you re-run it after this.


---
## v11 PATCH — fix `find_mask_for_image()` for `_mask`-suffixed filenames

Confirmed against a `tree` export: PolypGen2021-Video stores masks with a `_mask` suffix,
not an identical stem —

```
images_seq1/0_endocv2021_positive_104.jpg
masks_seq1/0_endocv2021_positive_104_mask.jpg
images_C1/100H0050.jpg
masks_C1/100H0050_mask.jpg
```

`find_mask_for_image()` (Cell 5) only ever tried an exact-stem match, so every one of the
29 v10 sub-datasets silently matched zero image→mask pairs → `no_valid_pairs` across
the board. This patch redefines the function to also try the `_mask` suffix, tried
**after** the exact match — so every dataset that already worked (CVC-300, Kvasir-SEG,
PolypDB, etc.) is unaffected. Run this cell, then **re-run Cell 15** (PolypGen2021-Video
positive sub-datasets) to get real Dice/IoU this time.


In [ ]:
# =================================================================
# v11 PATCH — find_mask_for_image(): support "<stem>_mask.<ext>" masks
# =================================================================
# Root cause (confirmed via tree export): PolypGen2021-Video masks are
# named "<stem>_mask.<ext>", not "<stem>.<ext>". The original function
# only tried the exact-stem match, so every images_seq*/masks_seq* and
# images_C*/masks_C* pair silently failed to match — hence the 29x
# "no_valid_pairs" in the v10 run.
#
# Exact-stem match is tried FIRST, unchanged, so every dataset that
# already worked (CVC-300, PolypDB, Kvasir-SEG, etc.) behaves identically.

def find_mask_for_image(img_path, mask_root):
    """Find the GT mask file for a given image path.

    Tries an exact-stem match first (original behavior, unchanged),
    then falls back to the "<stem>_mask.<ext>" convention used by
    PolypGen2021-Video's images_seq*/masks_seq* and images_C*/masks_C*
    folders, before doing a slower recursive walk with both patterns.
    """
    stem = Path(img_path).stem
    candidate_stems = [stem, f"{stem}_mask"]

    for cand_stem in candidate_stems:
        for ext in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
            candidate = os.path.join(mask_root, cand_stem + ext)
            if os.path.exists(candidate):
                return candidate

    # Search recursively, matching either the exact stem or "<stem>_mask"
    for root, _, files in os.walk(mask_root):
        for f in files:
            if Path(f).stem in candidate_stems:
                return os.path.join(root, f)
    return None


print("find_mask_for_image() patched to support '_mask'-suffixed filenames ✓")
print("Now re-run Cell 15 (PolypGen2021-Video positive sub-datasets) to pick up real pairs.")


In [ ]:
# =================================================================
# v10 PATCH — PolypGen2021-Video: real image+mask pairs
# =================================================================
# Root cause (confirmed via a full `tree` export of /kaggle/input):
# auto_find_img_mask_dirs() only matches a mask folder named EXACTLY
# "masks"/"mask"/"gt"/etc. The real folders here are suffixed
# ("masks_seq1", "masks_C1"), so every pair was silently skipped.
# Fix: enumerate the known suffix patterns explicitly instead of
# relying on the generic auto-detector.

PGV_ROOT_KEY = "polypgen20021-video"
_pgv_root = RESOLVED.get(PGV_ROOT_KEY)

PGV_SUBROOT = "PolypGen2021_MultiCenterData_v3"

pgv_positive_sub_datasets = []

# ── 23 per-sequence frame/mask pairs ─────────────────────────
for n in range(1, 24):
    pgv_positive_sub_datasets.append({
        "name":     f"PolypGen2021-Video/seq{n}",
        "img_sub":  f"{PGV_SUBROOT}/sequenceData/positive/seq{n}/images_seq{n}",
        "mask_sub": f"{PGV_SUBROOT}/sequenceData/positive/seq{n}/masks_seq{n}",
    })

# ── 6 per-center frame/mask pairs ───────────────────────────
for c in range(1, 7):
    pgv_positive_sub_datasets.append({
        "name":     f"PolypGen2021-Video/C{c}",
        "img_sub":  f"{PGV_SUBROOT}/data_C{c}/images_C{c}",
        "mask_sub": f"{PGV_SUBROOT}/data_C{c}/masks_C{c}",
    })

DATASET_MAP["polypgen20021-video"]["sub_datasets"] = pgv_positive_sub_datasets
DATASET_MAP["polypgen20021-video"]["role"] = "image_multi"  # now has real GT masks

print(f"PolypGen2021-Video root resolved: {_pgv_root}")
print(f"Registered {len(pgv_positive_sub_datasets)} image+mask sub-datasets "
      f"(23 sequences + 6 centers)\n")

if _pgv_root:
    print("Evaluating PolypGen2021-Video positive sub-datasets ...")
    for sub in pgv_positive_sub_datasets:
        run_image_ds(sub["name"], "polypgen20021-video", sub["img_sub"], sub["mask_sub"])
else:
    print("SKIP: polypgen20021-video not attached this run.")


In [ ]:
# =================================================================
# v10 PATCH — PolypGen2021-Video negative-control frames
# =================================================================
# sequenceData/negativeOnly/seq{1..23}_neg are flat .jpg frames with
# NO masks and NO polyps — a genuine specificity/false-positive check,
# same idea as LDPolyp-NoPolyp but stored as loose images, not a video.

def evaluate_image_negative_control(name, img_dirs):
    """
    Run YOLO (fallback=False, matching the v9 video-negative fix) over a
    set of image directories known to contain NO polyps, and report the
    false-positive rate / specificity.
    """
    total_frames, fp_frames = 0, 0
    for img_dir in img_dirs:
        if not os.path.isdir(img_dir):
            continue
        img_files = sorted([f for f in os.listdir(img_dir)
                             if Path(f).suffix.lower() in IMG_EXTS])
        for fname in img_files:
            img_bgr = cv2.imread(os.path.join(img_dir, fname))
            if img_bgr is None:
                continue
            pred_mask, n_boxes, mode = run_yolo_segment(
                img_bgr, YOLO_MODEL, SEG_MODEL, fallback=False
            )
            total_frames += 1
            if (pred_mask > 127).any():
                fp_frames += 1

    if total_frames == 0:
        return {"name": name, "status": "skip", "reason": "no negative-control frames found"}

    fpr = round(fp_frames / total_frames, 4)
    return {
        "name": name, "type": "video_negative",
        "status": "done",
        "n_videos": len(img_dirs),
        "n_frames_sampled": total_frames,
        "false_positive_frames": fp_frames,
        "false_positive_rate": fpr,
        "specificity": round(1.0 - fpr, 4),
        "is_negative": True,
    }


if _pgv_root:
    neg_dirs = [
        smart_join(_pgv_root, f"{PGV_SUBROOT}/sequenceData/negativeOnly/seq{n}_neg")
        for n in range(1, 24)
    ]
    print("Evaluating PolypGen2021-Video negative control (23 sequences) ...")
    r = evaluate_image_negative_control("PolypGen2021-Video-NoPolyp", neg_dirs)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        print(f"  FPR={r['false_positive_rate']:.4f}  specificity={r['specificity']:.4f}  "
              f"n_frames={r['n_frames_sampled']}")
    else:
        print(f"  SKIP: {r.get('reason','')}")
else:
    print("SKIP: polypgen20021-video not attached this run.")

print("\n✓ v10 patch evaluations complete — re-run Cell 11 to fold these into the comparison table.")
